# NumPy: rzadziej używane funkcje, które warto znać

**Problem:** pandas jest zbudowany na NumPy, ale wiele przydatnych funkcji NumPy zostaje w cieniu, bo pandas ma "swój" odpowiednik (`.clip()`, `.quantile()`, `.diff()`). Część z nich to czysta duplikacja — ale kilka z nich jest **szybszych, elastyczniejszych albo robi coś, czego pandas w ogóle nie ma wprost** (np. średnia ważona).

**Porównanie:** ta notatka nie powtarza tego, co już jest w innych notatkach o pandas (`groupby`, `pivot`, tekst) — skupia się na funkcjach NumPy jako **uzupełnieniu** warsztatu, z jawnym wskazaniem, gdzie mają przewagę nad odpowiednikiem pandas, a gdzie to tylko inny zapis tego samego.

**Kiedy stosować:** gdy operujesz na surowych tablicach (`np.ndarray`) bez potrzeby pełnego `DataFrame`; gdy potrzebna wektoryzowana logika warunkowa szybsza niż `.apply()`; gdy pandas nie ma wprost tego, co potrzebujesz (średnia ważona, partial sort).

## Setup

In [ ]:
import numpy as np
import pandas as pd

sales = np.array([1200, 800, 1500, 300, 2200, 950, 50, 1800])
units = np.array([12, 8, 15, 3, 22, 9, 1, 18])
region = np.array(["North", "North", "South", "South", "East", "East", "West", "West"])

sales, units, region

## Sekcja 1 — `np.where`: warunkowe przypisanie

Z **trzema** argumentami: `(warunek, jeśli_prawda, jeśli_fałsz)` — dokładnie jak `IF()` w Excelu, wektoryzowane. Z **jednym** argumentem (samym warunkiem) robi coś zupełnie innego — patrz Pułapka 1.

In [ ]:
np.where(sales > 1000, "wysoka", "niska")

## Sekcja 2 — `np.select`: wiele warunków naraz

Zagnieżdżanie `np.where` w `np.where` przy więcej niż dwóch kategoriach szybko robi się nieczytelne. `np.select(warunki, wybory, default=)` przyjmuje LISTĘ warunków i LISTĘ wyborów tej samej długości — pierwszy pasujący warunek wygrywa, `default` obsługuje przypadek, gdy żaden nie pasuje (patrz też Pułapka 3 — pomijanie `default` bywa niebezpieczne).

In [ ]:
conditions = [sales < 500, sales < 1500, sales >= 1500]
choices = ["niska", "średnia", "wysoka"]

np.select(conditions, choices, default="brak")

## Sekcja 3 — `np.logical_and`/`or`/`not`/`xor`

Dla dwóch warunków `and`/`or`/`not` to dokładnie to samo co operatory `&`/`|`/`~` — kwestia czytelności. Prawdziwa przewaga ujawnia się przy **więcej niż dwóch** warunkach naraz: `np.logical_and.reduce([...])` przyjmuje listę dowolnej długości, bez zagnieżdżania nawiasów.

**`np.logical_xor` jest inne od pozostałych** — nie ma bezpośredniego odpowiednika w jednym operatorze Pythona (`^` działa, ale mało kto o tym pamięta, a na typach innych niż bool/int bywa mylące). `xor` zwraca `True` tylko wtedy, gdy warunki **się nie zgadzają** — dokładnie jeden z dwóch jest prawdziwy, nie oba i nie żaden. To naturalne narzędzie do wykrywania ROZBIEŻNOŚCI między dwiema niezależnymi regułami, a nie tylko "czy któraś zadziałała".

In [ ]:
# Dla dwóch warunków - identyczny efekt co operator &
print(np.logical_and(sales > 500, units > 10))
print((sales > 500) & (units > 10))

# Dla wielu warunków naraz - .reduce() zamiast łańcucha nawiasów
many_conditions = [sales > 100, units > 5, sales < 2000]
print(np.logical_and.reduce(many_conditions))

### `np.logical_xor` w praktyce: dwie niezależne metody wykrywania anomalii, które się NIE zgadzają

Wyobraź sobie dwa niezależne "czujniki" podejrzanych transakcji: metoda A flaguje niską sprzedaż, metoda B flaguje niską liczbę jednostek. Interesują Cię trzy różne zbiory:
- **`logical_and`** — obie metody się zgadzają (najbardziej pewne anomalie, ale mało liczne).
- **`logical_or`** — zgłosiła choć jedna (pełny, szerszy zbiór podejrzanych).
- **`logical_xor`** — metody się NIE zgadzają — jedna zgłosiła alarm, druga nie. To właśnie te wiersze są najciekawsze do ręcznego audytu: albo jedna metoda ma fałszywy alarm, albo wykryła coś, czego druga nie widzi.

In [ ]:
flag_low_sales = sales < 400      # metoda A: podejrzanie niska sprzedaż
flag_low_units = units < 10       # metoda B: podejrzanie mało sprzedanych sztuk

print(f"Metoda A (sales < 400):  {flag_low_sales}")
print(f"Metoda B (units < 10):   {flag_low_units}")
print()
print(f"AND - obie się zgadzają:      {np.logical_and(flag_low_sales, flag_low_units)}")
print(f"XOR - metody się NIE zgadzają: {np.logical_xor(flag_low_sales, flag_low_units)}")
print(f"OR  - choć jedna zgłosiła:     {np.logical_or(flag_low_sales, flag_low_units)}")
print()
print("Wiersze do ręcznego audytu (rozbieżność metod):")
print(np.where(np.logical_xor(flag_low_sales, flag_low_units))[0])

### `np.logical_not`

Identyczne z operatorem `~`, ale bywa czytelniejsze w łańcuchu `.reduce()` albo gdy warunek sam jest wynikiem złożonego wyrażenia, które i tak trzeba trzymać w zmiennej.

In [ ]:
print(np.logical_not(flag_low_sales))
print(~flag_low_sales)  # dokładnie to samo

## Sekcja 4 — `np.maximum`/`np.minimum`: element po elemencie (NIE mylić z `np.max`/`np.min`)

To dwie zupełnie różne funkcje o bardzo podobnych nazwach:
- `np.maximum(a, b)` — porównuje DWIE tablice element-po-elemencie, zwraca tablicę tej samej długości.
- `np.max(a)` — REDUKUJE jedną tablicę do pojedynczej liczby.

`np.maximum`/`minimum` to naturalny sposób na "przynajmniej tyle" / "co najwyżej tyle" względem innej kolumny (nie stałej — do tego służy `np.clip`, Sekcja 5).

In [ ]:
target = np.array([1000, 1000, 1000, 1000, 2000, 1000, 1000, 2000])

print("np.maximum(sales, target) - element po elemencie:")
print(np.maximum(sales, target))

print("\nnp.max(sales) - REDUKCJA do jednej liczby:")
print(np.max(sales))

### `np.maximum.reduce`: to samo, ale dla WIĘCEJ NIŻ dwóch tablic naraz

`np.maximum(a, b)` przyjmuje dokładnie dwie tablice. Gdy porównywanych "kandydatów" jest więcej (np. trzy różne metody prognozy i chcesz wziąć najbardziej ostrożną/najwyższą z nich, element po elemencie), `np.maximum.reduce([...])` robi to w jednym wywołaniu, bez zagnieżdżania `np.maximum(np.maximum(a, b), c)`.

In [ ]:
forecast_method1 = np.array([100, 200, 150, 300])
forecast_method2 = np.array([120, 180, 160, 290])
forecast_method3 = np.array([110, 210, 140, 310])

# Najbardziej ostrożna (najwyższa) prognoza spośród trzech metod, element po elemencie
np.maximum.reduce([forecast_method1, forecast_method2, forecast_method3])

## Sekcja 5 — `np.clip`: ograniczenie do zakresu

W odróżnieniu od `np.maximum`/`minimum` (porównanie z inną tablicą), `np.clip(a, min, max)` ogranicza wartości do STAŁEGO zakresu — dokładny odpowiednik `Series.clip()` w pandas, tu na surowej tablicy.

In [ ]:
np.clip(sales, 500, 1500)

## Sekcja 6 — `np.quantile`/`np.percentile`: więcej kontroli niż pandas

Różnica między nimi to tylko skala (`quantile` 0–1, `percentile` 0–100). Przewaga nad `Series.quantile()`: parametr `method=` (dawniej `interpolation=`) z kilkoma strategiami — istotne przy małych/dyskretnych zbiorach, gdzie "dokładny" kwantyl nie trafia w żadną rzeczywistą obserwację.

In [ ]:
print(f"quantile (skala 0-1): {np.quantile(sales, 0.25)}")
print(f"percentile (skala 0-100): {np.percentile(sales, 25)}")
print()
for method in ["linear", "lower", "higher", "nearest", "midpoint"]:
    print(f"method='{method}': {np.quantile(sales, 0.25, method=method)}")

## Sekcja 7 — `np.digitize`: binowanie (alternatywa dla `pd.cut`)

Zwraca INDEKS przedziału, do którego trafia każda wartość — szybsze niż `pd.cut`, gdy potrzebujesz tylko numeru binu (bez pełnego obiektu `Categorical`), np. jako wejście do dalszej logiki wektoryzowanej.

In [ ]:
bins = [0, 500, 1000, 1500, 2000, 3000]
bin_idx = np.digitize(sales, bins)
labels = ["0-500", "500-1000", "1000-1500", "1500-2000", "2000-3000"]

print("Wartości:      ", sales)
print("Indeksy binów: ", bin_idx)
print("Etykiety:      ", [labels[i - 1] for i in bin_idx])

## Sekcja 8 — `np.searchsorted`: szybkie wyszukiwanie w posortowanej tablicy

Zwraca pozycję, na której dana wartość zostałaby wstawiona, żeby zachować sortowanie. **Trik:** to naturalny sposób na "VLOOKUP z przybliżeniem w dół" — np. dopasowanie ceny do progu ilościowego, bez pisania pętli ani `.apply()`.

In [ ]:
# Trik: cena za sztukę zależna od progu zamówionej ilości (im więcej, tym taniej)
qty_breaks = np.array([0, 10, 50, 100])
prices = np.array([100, 90, 80, 70])
order_qty = np.array([5, 25, 75, 150])

price_idx = np.searchsorted(qty_breaks, order_qty, side="right") - 1
matched_prices = prices[price_idx]

print(f"Zamówienia:     {order_qty}")
print(f"Dopasowane ceny: {matched_prices}")

## Sekcja 9 — `np.average`: średnia WAŻONA (czego pandas nie ma wprost)

`Series.mean()` nie przyjmuje wag. `np.average(a, weights=...)` tak — przydatne np. do średniej ceny ważonej ilością sprzedanych sztuk, zamiast zwykłej średniej z cen.

In [ ]:
weights = np.array([1, 1, 2, 1, 3, 1, 1, 2])

print(f"Zwykła średnia:  {np.mean(sales):.1f}")
print(f"Średnia ważona:  {np.average(sales, weights=weights):.1f}")

## Sekcja 10 — `np.diff`: różnice, z kontrolą rzędu

Odpowiednik `Series.diff()`, ale z parametrem `n=` do różnic WYŻSZEGO rzędu — `n=2` liczy różnicę różnic, czyli przyspieszenie zmiany (druga pochodna dyskretna), nie tylko samą zmianę.

In [ ]:
print(f"Wartości:              {sales}")
print(f"Różnica 1. rzędu:      {np.diff(sales)}")
print(f"Różnica 2. rzędu (n=2): {np.diff(sales, n=2)}")

## Sekcja 11 — `np.unique` z `return_counts`

Podobne do `value_counts()`, ale działa na surowej tablicy (nie wymaga `Series`) i zwraca gotową PARĘ tablic (unikalne wartości + liczności) zamiast obiektu z własnym indeksem — czasem wygodniejsze do dalszych operacji wektoryzowanych.

In [ ]:
values, counts = np.unique(region, return_counts=True)
dict(zip(values, counts))

## Sekcja 12 — `np.argpartition`: szybkie top-N BEZ pełnego sortowania

Gdy potrzebujesz tylko "N największych wartości" (nie ich dokładnej kolejności), pełne sortowanie (`np.argsort`) robi więcej pracy niż trzeba — sortuje WSZYSTKO, żeby dobrać się do kilku ostatnich. `np.argpartition` gwarantuje tylko, że N największych znajdzie się na właściwym końcu tablicy, bez sortowania ich między sobą — i jest za to szybsze.

In [ ]:
import time

rng = np.random.default_rng(2)
big = rng.integers(0, 1_000_000, 1_000_000)
k = 5

start = time.perf_counter()
top_k_full_sort = np.argsort(big)[-k:]
t_sort = time.perf_counter() - start

start = time.perf_counter()
top_k_partition = np.argpartition(big, -k)[-k:]
t_partition = time.perf_counter() - start

print(f"Pełne argsort:           {t_sort*1000:.2f} ms")
print(f"argpartition (top-{k}):    {t_partition*1000:.2f} ms")
print(f"Różnica: {t_sort/t_partition:.1f}x szybciej")
print(f"Te same wartości top-{k}: {sorted(big[top_k_full_sort]) == sorted(big[top_k_partition])}")

## Sekcja 13 — Benchmark: `np.select` vs `.apply()` z funkcją warunkową

Ten sam motyw co w innych notatkach: wektoryzowana alternatywa dla logiki `if/elif` w `.apply()`.

In [ ]:
n = 300_000
big_sales = rng.integers(0, 3000, n)
df_big = pd.DataFrame({"sales": big_sales})


def categorize(x):
    if x < 500:
        return "niska"
    elif x < 1500:
        return "średnia"
    return "wysoka"


start = time.perf_counter()
result_apply = df_big["sales"].apply(categorize)
t_apply = time.perf_counter() - start

start = time.perf_counter()
arr = df_big["sales"].to_numpy()
result_select = np.select([arr < 500, arr < 1500], ["niska", "średnia"], default="wysoka")
t_select = time.perf_counter() - start

print(f".apply(funkcja) z if/elif: {t_apply:.4f}s")
print(f"np.select (wektoryzowane): {t_select:.4f}s")
print(f"Różnica: {t_apply/t_select:.1f}x szybciej")
print(f"Wyniki identyczne: {(result_apply.to_numpy() == result_select).all()}")

## Sekcja 14 — `np.interp`: interpolacja liniowa

Dla znanych par punktów `(x, y)` oblicza wartość `y` w dowolnym nowym punkcie `x`, liniowo między najbliższymi znanymi punktami. Przydatne do uzupełniania brakujących wartości na osi ciągłej (np. szacowanie wartości między znanymi pomiarami) albo przeskalowania jednej skali na drugą.

**Warto wiedzieć:** wartości spoza zakresu znanych `x` nie są ekstrapolowane — domyślnie "przyklejają się" do wartości na brzegu zakresu.

In [ ]:
known_x = np.array([1, 3, 6, 10])
known_y = np.array([100, 300, 550, 900])
new_x = np.array([2, 4, 5, 8])

print(f"Interpolowane wartości: {np.interp(new_x, known_x, known_y)}")
print()

outside_range = np.array([-5, 20])
print(f"Poza zakresem (-5, 20) - przyklejone do brzegów, NIE ekstrapolowane: {np.interp(outside_range, known_x, known_y)}")

## Sekcja 15 — Rodzina `np.nan*`: statystyki odporne na braki danych

To jedna z najbardziej niedocenianych rodzin funkcji w NumPy. **Zwykłe** `np.mean()`, `np.sum()`, `np.max()` itd. propagują `NaN` do CAŁEGO wyniku — jeden brak w tablicy 10 000 elementów wystarczy, żeby zepsuć wszystko. Pandas robi to za Ciebie automatycznie (`Series.mean()` ignoruje `NaN` domyślnie) — ale na surowej tablicy NumPy trzeba jawnie sięgnąć po wariant `nan*`: `nanmean`, `nansum`, `nanmax`, `nanmin`, `nanstd`, `nanargmax`, `nanargmin`, `nanpercentile`.

In [ ]:
sales_with_gaps = np.array([1200, 800, np.nan, 300, 2200, np.nan, 50, 1800])

print("Zwykłe funkcje - JEDEN NaN psuje CAŁY wynik:")
print(f"  np.mean: {np.mean(sales_with_gaps)}")
print(f"  np.sum:  {np.sum(sales_with_gaps)}")
print(f"  np.max:  {np.max(sales_with_gaps)}")
print()
print("Wariant nan* - ignoruje braki, liczy na tym, co realnie dostępne:")
print(f"  np.nanmean:       {np.nanmean(sales_with_gaps):.2f}")
print(f"  np.nansum:        {np.nansum(sales_with_gaps)}")
print(f"  np.nanmax:        {np.nanmax(sales_with_gaps)}")
print(f"  np.nanargmax:     {np.nanargmax(sales_with_gaps)} (indeks maksimum, z pominięciem NaN)")
print(f"  np.nanpercentile: {np.nanpercentile(sales_with_gaps, 25)}")
print(f"  np.nanstd:        {np.nanstd(sales_with_gaps):.2f}")

**Pułapka wewnątrz triku:** jeśli CAŁA tablica (albo cały wycinek, np. jedna grupa po `groupby`) to same `NaN`, `nanargmax`/`nanmax` nie zwracają `NaN` ani `0` — rzucają wyjątek, bo "maksimum z niczego" nie ma sensownej odpowiedzi. Warto to przechwycić, jeśli operujesz na wielu podgrupach danych, gdzie któraś może być pusta.

In [ ]:
all_missing = np.array([np.nan, np.nan, np.nan])
try:
    np.nanargmax(all_missing)
except ValueError as e:
    print(f"Błąd na samych NaN: {e}")

## Sekcja 16 — `np.bincount`: błyskawiczne zliczanie zakodowanych kategorii

Gdy kategorie są już zakodowane jako małe, nieujemne liczby całkowite (typowy wynik `pd.factorize()` albo ręcznego mapowania), `np.bincount()` liczy wystąpienia szybciej niż `np.unique(..., return_counts=True)` — nie musi niczego sortować, tylko zlicza do tablicy o długości `max(wartość) + 1`.

In [ ]:
# Regiony zakodowane jako int: 0=North, 1=South, 2=East, 3=West
region_codes = np.array([0, 0, 1, 1, 2, 2, 2, 3])
counts = np.bincount(region_codes)

region_names = ["North", "South", "East", "West"]
for name, count in zip(region_names, counts):
    print(f"{name}: {count}")

## Sekcja 17 — `np.corrcoef` / `np.cov`: szybka korelacja bez pełnego `DataFrame`

Gdy masz tylko dwie (albo więcej) surowe tablice liczbowe i chcesz jeden liczbowy wynik korelacji, `np.corrcoef()` jest szybsze niż budowanie `DataFrame` tylko po to, żeby wywołać `.corr()`. Zwraca **macierz** — dla dwóch zmiennych interesująca wartość siedzi poza przekątną, na pozycji `[0, 1]` (albo `[1, 0]`, symetrycznie).

In [ ]:
correlation_matrix = np.corrcoef(sales, units)
print("Pełna macierz korelacji:")
print(correlation_matrix)
print()
print(f"Korelacja sales-units: {correlation_matrix[0, 1]:.4f}")

## Sekcja 18 — `np.isclose` / `np.allclose`: bezpieczne porównanie liczb zmiennoprzecinkowych

`==` na liczbach `float` jest zawodne — błędy zaokrągleń reprezentacji binarnej sprawiają, że matematycznie równe wartości bywają "nierówne" dla komputera. `np.isclose()` (element po elemencie) i `np.allclose()` (jeden wynik dla całej tablicy) porównują z tolerancją zamiast dosłownie.

In [ ]:
a = 0.1 + 0.2
b = 0.3

print(f"0.1 + 0.2 == 0.3           -> {a == b}  (klasyczna pułapka reprezentacji float)")
print(f"np.isclose(0.1+0.2, 0.3)   -> {np.isclose(a, b)}")
print()

arr_a = np.array([1.0000001, 2.0, 3.0])
arr_b = np.array([1.0, 2.0, 3.0])
print(f"np.allclose(arr_a, arr_b) -> {np.allclose(arr_a, arr_b)}  (różnica poniżej domyślnej tolerancji)")

### Powiązana pułapka: `np.round` NIE zaokrągla "po szkolnemu"

NumPy (podobnie jak Python) stosuje **bankers' rounding** (zaokrąglanie do najbliższej liczby parzystej) dla dokładnych wartości `.5` — nie zawsze w górę, jak uczono w szkole. To celowa decyzja projektowa (redukuje systematyczne obciążenie przy zaokrąglaniu wielu wartości), ale potrafi zaskoczyć.

In [ ]:
half_values = np.array([0.5, 1.5, 2.5, 3.5, 4.5])
print(f"Wartości:           {half_values}")
print(f"np.round():          {np.round(half_values)}")
print("Oczekiwane 'szkolne' (zawsze w górę): [1. 2. 3. 4. 5.] - NumPy tak NIE działa")
print("Zamiast tego zaokrągla do najbliższej PARZYSTEJ: 0.5->0, 1.5->2, 2.5->2, 3.5->4, 4.5->4")

## Sekcja 19 — Broadcasting: dlaczego operacje wektorowe w ogóle działają

Broadcasting to reguły, według których NumPy dopasowuje tablice o RÓŻNYCH kształtach do wspólnej operacji, bez pisania pętli. Rozumienie tego mechanizmu tłumaczy, dlaczego np. `macierz - wektor` w ogóle ma sens i co dokładnie się dzieje.

In [ ]:
sales_matrix = np.array([
    [1200, 800, 1500],   # region 1: sprzedaż w 3 kolejnych miesiącach
    [900, 950, 1100],    # region 2
])
print(f"Macierz sprzedaży {sales_matrix.shape} (2 regiony x 3 miesiące):")
print(sales_matrix)

In [ ]:
# Broadcasting SKALARA - jedna liczba "rozgłoszona" do każdej komórki macierzy
sales_matrix - 1000

In [ ]:
# Broadcasting WEKTORA WIERSZOWEGO (3,) - rozgłoszony do KAŻDEGO wiersza macierzy
monthly_avg = np.array([1050, 875, 1300])  # średnia w każdym z 3 miesięcy
sales_matrix - monthly_avg

In [ ]:
# Broadcasting WEKTORA KOLUMNOWEGO (2,1) - rozgłoszony do KAŻDEJ kolumny macierzy
regional_avg = np.array([[1166.7], [983.3]])  # średnia w każdym z 2 regionów
np.round(sales_matrix - regional_avg, 1)

In [ ]:
# Kształty NIEZGODNE ze sobą - jawny błąd, nie cichy błędny wynik
try:
    sales_matrix - np.array([1, 2])  # (2,3) i (2,) - nie da się dopasować
except ValueError as e:
    print(f"Błąd broadcastingu: {e}")

**Reguła w skrócie:** porównując kształty od PRAWEJ strony, wymiary muszą być albo identyczne, albo jeden z nich musi być `1` (wtedy jest "rozciągany"). `(2,3)` i `(3,)` pasują (3=3). `(2,3)` i `(2,1)` pasują (3 vs 1 rozciągnięte, 2=2). `(2,3)` i `(2,)` NIE pasują — porównując od prawej: 3 vs 2, żadna z tych liczb nie jest 1.

## Sekcja 20 — `np.repeat` vs `np.tile`: dwa różne wzorce powtarzania

Nazwy nic nie sugerują same z siebie, a efekt jest zupełnie inny: `repeat` powtarza KAŻDY element z osobna (świetne np. do rozwinięcia jednej wartości na grupę wierszy), `tile` powtarza CAŁĄ tablicę jako blok (świetne do budowania powtarzalnych wzorców, np. cyklicznych etykiet).

In [ ]:
small = np.array([1, 2, 3])

print(f"np.repeat(small, 2) - każdy element powtórzony:  {np.repeat(small, 2)}")
print(f"np.tile(small, 2)   - cała tablica powtórzona:    {np.tile(small, 2)}")

## Sekcja 21 — `np.polyfit` / `np.poly1d`: szybkie dopasowanie trendu

Dopasowanie prostej (albo wielomianu wyższego stopnia) do punktów danych bez importowania osobnej biblioteki do regresji — przydatne do szybkiego oszacowania trendu i prostej ekstrapolacji, gdy pełny model statystyczny to przesada.

In [ ]:
months_num = np.array([1, 2, 3, 4, 5, 6])
sales_trend = np.array([1000, 1050, 1180, 1200, 1350, 1400])

# deg=1 - dopasowanie prostej (nachylenie, wyraz wolny)
slope, intercept = np.polyfit(months_num, sales_trend, deg=1)
print(f"Nachylenie trendu: {slope:.1f} PLN/miesiąc")
print(f"Wyraz wolny:       {intercept:.1f}")

trend_line = np.poly1d([slope, intercept])
print(f"\nPrognoza na miesiąc 7 (ekstrapolacja): {trend_line(7):.1f}")
print(f"Wartości dopasowane do znanych miesięcy: {np.round(trend_line(months_num), 1)}")

## Sekcja 14 — Pułapki

### Pułapka 1 — `np.where` z JEDNYM argumentem zwraca krotkę INDEKSÓW, nie wartości

To nie jest skrócona forma trzyargumentowego `np.where` — to zupełnie inna funkcja o tej samej nazwie. Bez `jeśli_prawda`/`jeśli_fałsz`, `np.where(warunek)` zachowuje się jak `np.nonzero()` — zwraca pozycje, na których warunek jest `True`, zapakowane w krotkę (po jednej tablicy na wymiar).

In [ ]:
result_3args = np.where(sales > 1000, "wysoka", "niska")
result_1arg = np.where(sales > 1000)

print(f"np.where(warunek, x, y) -> tablica wartości: {result_3args}")
print(f"np.where(warunek)       -> {type(result_1arg).__name__} indeksów: {result_1arg}")

### Pułapka 2 — `np.maximum` vs `np.max`: nazwa różni się jedną literą, znaczenie jest zupełnie inne

Podsumowanie z Sekcji 4: `maximum`/`minimum` (l. mnoga formy, dwie tablice) porównuje element-po-elemencie; `max`/`min` (l. pojedyncza) redukuje jedną tablicę do liczby. Użycie jednego zamiast drugiego albo rzuci błąd (zła liczba argumentów), albo — gorzej — zwróci coś, co wygląda sensownie, ale znaczy coś innego.

### Pułapka 3 — `np.select` bez `default`: brakująca wartość cicho staje się `0`

Gdy żaden warunek nie pasuje (np. wartość źródłowa to `NaN`), a `choicelist` jest liczbowa, domyślny `default` to `0` — nie `NaN`, nie błąd, nie ostrzeżenie. Wygląda jak realna, policzona wartość zerowa, a w rzeczywistości oznacza "żaden warunek nie zadziałał".

In [ ]:
sales_with_nan = np.array([1200, 800, np.nan, 300])
conditions = [sales_with_nan < 500, sales_with_nan >= 500]
choices_numeric = [sales_with_nan * 0.1, sales_with_nan * 0.2]  # np. dwie różne stawki prowizji

print("Bez jawnego default - NaN cicho zamienia się w 0.0:")
print(np.select(conditions, choices_numeric))

print("\nZ jawnym default=np.nan - poprawnie widać brak dopasowania:")
print(np.select(conditions, choices_numeric, default=np.nan))

## Podsumowanie

| Zadanie | Funkcja NumPy | Odpowiednik / uwaga pandas |
|---|---|---|
| Warunkowe przypisanie (if/else wektorowo) | `np.where(warunek, x, y)` | `Series.where()`/`.mask()` robią podobnie, ale odwrotnie sparametryzowane |
| Wiele warunków naraz | `np.select(warunki, wybory, default=)` | brak bezpośredniego odpowiednika — czytelniejsze niż zagnieżdżone `np.where` |
| Wiele warunków logicznych naraz | `np.logical_and.reduce([...])` | brak odpowiednika 1:1 |
| Dokładnie jeden z dwóch warunków (rozbieżność metod) | `np.logical_xor(a, b)` | brak odpowiednika 1:1 |
| Większa/mniejsza z DWÓCH tablic, element po elemencie | `np.maximum`/`np.minimum` | brak wprost — zwykle robi się przez `.where()`/`np.where` |
| To samo dla WIĘCEJ niż dwóch tablic naraz | `np.maximum.reduce([...])` | brak odpowiednika 1:1 |
| Ograniczenie do zakresu | `np.clip(a, min, max)` | `Series.clip()` — to samo |
| Kwantyl z kontrolą metody interpolacji | `np.quantile(a, q, method=)` | `Series.quantile()` ma mniej opcji `method` |
| Binowanie do indeksów przedziałów | `np.digitize(a, bins)` | `pd.cut()` daje więcej (etykiety, `Categorical`), `digitize` jest szybszy |
| Przybliżone wyszukiwanie w posortowanej tablicy | `np.searchsorted(sorted_a, values)` | brak wprost — przydatne do "lookup po progu" |
| Średnia ważona | `np.average(a, weights=)` | **brak w pandas wprost** |
| Różnica wyższego rzędu | `np.diff(a, n=)` | `Series.diff()` nie ma parametru `n` |
| Unikalne wartości + liczności | `np.unique(a, return_counts=True)` | `Series.value_counts()` — podobne, inny kształt wyniku |
| Top-N bez pełnego sortowania | `np.argpartition(a, -k)[-k:]` | `Series.nlargest()` — wygodniejsze, ale wolniejsze na dużych danych |
| Interpolacja liniowa między znanymi punktami | `np.interp(nowe_x, znane_x, znane_y)` | brak odpowiednika 1:1 (`Series.interpolate()` działa inaczej — po pozycji, nie po wartości x) |
| Statystyki odporne na `NaN` na surowej tablicy | `np.nanmean`/`nansum`/`nanmax`/`nanargmax`/`nanpercentile` | pandas robi to automatycznie na `Series` |
| Szybkie zliczanie zakodowanych kategorii (int) | `np.bincount(a)` | `Series.value_counts()` — ogólniejsze, wolniejsze |
| Korelacja bez budowania `DataFrame` | `np.corrcoef(a, b)` | `DataFrame.corr()` |
| Bezpieczne porównanie liczb zmiennoprzecinkowych | `np.isclose`/`np.allclose` | brak odpowiednika 1:1 |
| Zaokrąglanie — UWAGA na bankers' rounding | `np.round` | `Series.round()` ma to samo zachowanie |
| Powtórzenie każdego elementu z osobna | `np.repeat(a, n)` | `Series.repeat(n)` — to samo |
| Powtórzenie całej tablicy jako bloku | `np.tile(a, n)` | brak odpowiednika 1:1 |
| Szybkie dopasowanie trendu liniowego | `np.polyfit(x, y, deg=1)` + `np.poly1d` | brak wprost — zwykle `statsmodels`/`sklearn` do pełnej regresji |

**Wniosek:** żadna z tych funkcji nie zastępuje pandas — to dopełnienie warsztatu tam, gdzie surowa tablica NumPy wystarczy (szybciej) albo gdzie pandas czegoś nie ma wprost (średnia ważona, wyższe różnice). Jak wszędzie w tym repo: `np.where`/`np.select` bez pełnej listy argumentów nie rzucają błędu — dają wynik, który wygląda poprawnie, dopóki nie porówna się go z oczekiwaniem.